# 신경망의 노드 가지치기를 위한 유전 알고리즘
(Neural Networks Node Pruning with Genetic Algorithm)

이 노트북은 논문 "신경망의 노드 가지치기를 위한 유전 알고리즘"을 구현한 코드입니다.

## 필요 라이브러리 설치 및 임포트

In [1]:
# 필요한 라이브러리 설치
!pip install ucimlrepo numpy pandas scikit-learn tensorflow

  Using cached numpy-2.0.2-cp39-cp39-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (19.5 MB)
  Using cached numpy-2.0.2-cp39-cp39-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (19.5 MB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/38.6 MB ? eta -:--:--  Downloading scipy-1.13.1-cp39-cp39-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (38.6 MB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.6/38.6 MB 46.6 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.6/38.6 MB 46.6 MB/s eta 0:00:0000:01
  Attempting uninstall: numpy
    Found existing installation: numpy 1.21.5
  Attempting uninstall: numpy
    Found existing installation: numpy 1.21.5
    Uninstalling numpy-1.21.5:
      Successfully uninstalled numpy-1.21.5
    Uninstalling numpy-1.21.5:
      Successfully uninstalled numpy-1.21.5
  Attempting uninstall: scipy
  Attempting uninstall: scipy
    Found existing installation: scipy 1.9.1
    Uninstalling scipy-1.9.1:
    Found existing installation: 

In [2]:
# 필요한 라이브러리 임포트
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
import random
import math
from ucimlrepo import fetch_ucirepo

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

## 데이터 불러오기 및 전처리

In [ ]:
# UCI 데이터셋 불러오기
# 1. Balance Scale 데이터셋
balance_scale = fetch_ucirepo(id=12)
X_balance = balance_scale.data.features 
y_balance = balance_scale.data.targets

# 2. Glass Identification 데이터셋
glass = fetch_ucirepo(id=42)
X_glass = glass.data.features
y_glass = glass.data.targets

# 3. Ionosphere 데이터셋
ionosphere = fetch_ucirepo(id=52)
X_ionosphere = ionosphere.data.features
y_ionosphere = ionosphere.data.targets

# 4. Iris 데이터셋
iris = fetch_ucirepo(id=53)
X_iris = iris.data.features
y_iris = iris.data.targets

# 5. Image Segmentation 데이터셋
segmentation = fetch_ucirepo(id=50)
X_segmentation = segmentation.data.features
y_segmentation = segmentation.data.targets

# 6. Zoo 데이터셋
zoo = fetch_ucirepo(id=111)
X_zoo = zoo.data.features
y_zoo = zoo.data.targets

# 데이터셋 정보 출력
datasets = {
    'balance_scale': (X_balance, y_balance),
    'glass': (X_glass, y_glass),
    'ionosphere': (X_ionosphere, y_ionosphere),
    'iris': (X_iris, y_iris),
    'segmentation': (X_segmentation, y_segmentation),
    'zoo': (X_zoo, y_zoo)
}

for name, (X, y) in datasets.items():
    print(f"{name} 데이터셋: 샘플 수={X.shape[0]}, 특징 수={X.shape[1]}, 클래스 수={len(np.unique(y))}")

## 데이터 전처리 함수

In [ ]:
def preprocess_data(X, y):
    """
    데이터를 전처리하는 함수
    
    Args:
        X: 입력 특징
        y: 타겟 레이블
        
    Returns:
        X_scaled: 정규화된 특징
        y_encoded: 원-핫 인코딩된 레이블
    """
    # 특징 정규화 (0-1 범위로)
    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(X)
    
    # 원-핫 인코딩 (필요 시)
    if len(np.unique(y)) > 2:
        # 다중 클래스 분류인 경우 원-핫 인코딩
        y_encoded = tf.keras.utils.to_categorical(y)
    else:
        # 이진 분류인 경우 그대로 사용
        y_encoded = y
    
    return X_scaled, y_encoded

## 신경망 모델 정의

In [ ]:
def create_neural_network(input_nodes, hidden_nodes, output_nodes):
    """
    다층 퍼셉트론 신경망 모델 생성
    
    Args:
        input_nodes: 입력층 노드 수
        hidden_nodes: 은닉층 노드 수
        output_nodes: 출력층 노드 수
        
    Returns:
        model: 신경망 모델
    """
    model = Sequential()
    model.add(Dense(hidden_nodes, input_dim=input_nodes, activation='relu'))
    
    # 출력층 활성화 함수 결정
    if output_nodes > 1:
        model.add(Dense(output_nodes, activation='softmax'))  # 다중 클래스 분류
        loss = 'categorical_crossentropy'
    else:
        model.add(Dense(output_nodes, activation='sigmoid'))  # 이진 분류
        loss = 'binary_crossentropy'
    
    model.compile(loss=loss, optimizer='adam', metrics=['accuracy'])
    return model

## 유전 알고리즘을 이용한 신경망 노드 가지치기 구현

In [ ]:
class NNPGA:
    def __init__(self, X, y, input_nodes, hidden_nodes, output_nodes, population_size=20, 
                 crossover_prob=1.0, mutation_prob=0.01, selection_pressure=0.25, max_generations=150):
        """
        NNPGA(Neural Networks Node Pruning with Genetic Algorithm) 초기화
        
        Args:
            X: 입력 데이터
            y: 타겟 데이터
            input_nodes: 입력층 노드 수
            hidden_nodes: 은닉층 노드 수
            output_nodes: 출력층 노드 수
            population_size: 해집단 크기
            crossover_prob: 교차 확률
            mutation_prob: 돌연변이 확률
            selection_pressure: 선택압 (q 값)
            max_generations: 최대 세대 수
        """
        self.X = X
        self.y = y
        self.input_nodes = input_nodes
        self.hidden_nodes = hidden_nodes
        self.output_nodes = output_nodes
        self.population_size = population_size
        self.crossover_prob = crossover_prob
        self.mutation_prob = mutation_prob
        self.selection_pressure = selection_pressure
        self.max_generations = max_generations
        
        # 가중치 저장을 위한 딕셔너리
        self.weights = {}
        
        # 초기 해집단 생성
        self.population = self.initialize_population()
        
        # 적합도 저장 리스트
        self.fitness_history = []
    
    def initialize_population(self):
        """
        초기 해집단 생성
        
        Returns:
            population: 초기 해집단
        """
        population = []
        
        for _ in range(self.population_size):
            # 모든 노드 활성화 상태로 초기화
            chromosome = np.ones(self.input_nodes + self.hidden_nodes, dtype=int)
            
            # 입력층 노드 가지치기 (8% 비율로 설정)
            pruning_rate = 0.08
            input_prune_count = int(self.input_nodes * pruning_rate)
            hidden_prune_count = int(self.hidden_nodes * pruning_rate)
            
            # 입력층 노드 가지치기
            input_indices = np.random.choice(self.input_nodes, input_prune_count, replace=False)
            for idx in input_indices:
                chromosome[idx] = 0
            
            # 은닉층 노드 가지치기
            hidden_indices = np.random.choice(range(self.input_nodes, self.input_nodes + self.hidden_nodes), 
                                             hidden_prune_count, replace=False)
            for idx in hidden_indices:
                chromosome[idx] = 0
            
            population.append(chromosome)
        
        return population
    
    def evaluate_fitness(self, chromosome, X_train, y_train, X_val, y_val):
        """
        염색체의 적합도 평가
        
        Args:
            chromosome: 염색체
            X_train: 훈련 데이터
            y_train: 훈련 데이터 레이블
            X_val: 검증 데이터
            y_val: 검증 데이터 레이블
            
        Returns:
            fitness: 적합도 (1/(1+RMSE))
            model: 훈련된 모델
        """
        # 입력층과 은닉층 노드 상태 확인
        active_input_nodes = np.where(chromosome[:self.input_nodes] == 1)[0]
        active_hidden_nodes = np.sum(chromosome[self.input_nodes:])
        
        if len(active_input_nodes) == 0 or active_hidden_nodes == 0:
            return 0, None  # 활성화된 노드가 없으면 적합도 0
        
        # 활성화된 입력 특징만 사용
        X_train_pruned = X_train[:, active_input_nodes]
        X_val_pruned = X_val[:, active_input_nodes]
        
        # 신경망 모델 생성
        model = Sequential()
        model.add(Dense(active_hidden_nodes, input_dim=len(active_input_nodes), activation='relu'))
        
        if isinstance(y_train, np.ndarray) and len(y_train.shape) > 1 and y_train.shape[1] > 1:
            # 다중 클래스 분류
            model.add(Dense(y_train.shape[1], activation='softmax'))
            loss = 'categorical_crossentropy'
        else:
            # 이진 분류
            model.add(Dense(1, activation='sigmoid'))
            loss = 'binary_crossentropy'
        
        model.compile(loss=loss, optimizer='adam', metrics=['accuracy'])
        
        # 조기 종료 설정
        early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
        
        # 모델 훈련
        model.fit(X_train_pruned, y_train, validation_data=(X_val_pruned, y_val), 
                 epochs=100, batch_size=32, callbacks=[early_stopping], verbose=0)
        
        # 검증 데이터에 대한 예측
        if isinstance(y_val, np.ndarray) and len(y_val.shape) > 1 and y_val.shape[1] > 1:
            # 다중 클래스 분류
            y_pred = model.predict(X_val_pruned)
            mse = mean_squared_error(y_val, y_pred)
        else:
            # 이진 분류
            y_pred = model.predict(X_val_pruned).flatten()
            mse = mean_squared_error(y_val, y_pred)
        
        rmse = np.sqrt(mse)
        fitness = 1 / (1 + rmse)
        
        return fitness, model
    
    def select_parent(self, fitness_values):
        """
        순위 기반 룰렛휠 선택 방법으로 부모 선택
        
        Args:
            fitness_values: 적합도 값 리스트
            
        Returns:
            selected_idx: 선택된 개체의 인덱스
        """
        # 적합도 순위 계산 (높은 적합도가 낮은 순위)
        ranked_indices = np.argsort(fitness_values)[::-1]
        
        # 순위 기반 선택 확률 계산
        q = self.selection_pressure
        selection_probs = [q * (1 - q) ** i for i in range(len(fitness_values))]
        
        # 정규화
        selection_probs = [selection_probs[i] for i in ranked_indices]
        sum_probs = sum(selection_probs)
        if sum_probs > 0:
            selection_probs = [p / sum_probs for p in selection_probs]
        else:
            # 모든 확률이 0인 경우 (드문 경우)
            selection_probs = [1.0 / len(fitness_values)] * len(fitness_values)
        
        # 룰렛휠 선택
        r = random.random()
        sum_prob = 0
        for i, prob in enumerate(selection_probs):
            sum_prob += prob
            if r <= sum_prob:
                return ranked_indices[i]
        
        # 기본값 (마지막 개체)
        return ranked_indices[-1]
    
    def crossover(self, parent1, parent2):
        """
        3점 교차 연산
        
        Args:
            parent1, parent2: 부모 염색체
            
        Returns:
            child: 자식 염색체
        """
        if random.random() > self.crossover_prob:
            return parent1.copy()
        
        # 3점 교차 지점 선택
        points = sorted(random.sample(range(1, len(parent1)), 3))
        
        # 교차 수행
        child = np.zeros_like(parent1)
        for i in range(len(parent1)):
            if i < points[0]:
                child[i] = parent1[i]
            elif i < points[1]:
                child[i] = parent2[i]
            elif i < points[2]:
                child[i] = parent1[i]
            else:
                child[i] = parent2[i]
        
        return child
    
    def mutation(self, chromosome):
        """
        돌연변이 연산
        
        Args:
            chromosome: 염색체
            
        Returns:
            mutated_chromosome: 돌연변이 된 염색체
        """
        mutated_chromosome = chromosome.copy()
        
        for i in range(len(chromosome)):
            if random.random() < self.mutation_prob:
                mutated_chromosome[i] = 1 - mutated_chromosome[i]  # 0->1, 1->0 반전
        
        # 가지치기 비율 유지 (8%)
        pruning_rate = 0.08
        
        # 입력층 노드 가지치기 비율 확인 및 조정
        input_active = np.sum(mutated_chromosome[:self.input_nodes])
        input_prune_target = int(self.input_nodes * (1 - pruning_rate))
        
        if input_active > input_prune_target:  # 더 많은 노드를 꺼야 함
            active_indices = np.where(mutated_chromosome[:self.input_nodes] == 1)[0]
            to_prune = random.sample(list(active_indices), int(input_active - input_prune_target))
            for idx in to_prune:
                mutated_chromosome[idx] = 0
        elif input_active < input_prune_target:  # 더 많은 노드를 켜야 함
            inactive_indices = np.where(mutated_chromosome[:self.input_nodes] == 0)[0]
            to_activate = random.sample(list(inactive_indices), min(int(input_prune_target - input_active), len(inactive_indices)))
            for idx in to_activate:
                mutated_chromosome[idx] = 1
        
        # 은닉층 노드 가지치기 비율 확인 및 조정
        hidden_active = np.sum(mutated_chromosome[self.input_nodes:])
        hidden_prune_target = int(self.hidden_nodes * (1 - pruning_rate))
        
        if hidden_active > hidden_prune_target:  # 더 많은 노드를 꺼야 함
            active_indices = np.where(mutated_chromosome[self.input_nodes:] == 1)[0] + self.input_nodes
            to_prune = random.sample(list(active_indices), int(hidden_active - hidden_prune_target))
            for idx in to_prune:
                mutated_chromosome[idx] = 0
        elif hidden_active < hidden_prune_target:  # 더 많은 노드를 켜야 함
            inactive_indices = np.where(mutated_chromosome[self.input_nodes:] == 0)[0] + self.input_nodes
            to_activate = random.sample(list(inactive_indices), min(int(hidden_prune_target - hidden_active), len(inactive_indices)))
            for idx in to_activate:
                mutated_chromosome[idx] = 1
        
        return mutated_chromosome
    
    def inherit_weights(self, child, parent1, parent2, p1_model, p2_model):
        """
        부모로부터 가중치 상속
        
        Args:
            child: 자식 염색체
            parent1, parent2: 부모 염색체
            p1_model, p2_model: 부모 모델
            
        Returns:
            inherited_weights: 상속받은 가중치
        """
        # 자식 모델에 필요한 활성화된 입력과 은닉 노드 확인
        child_active_inputs = np.where(child[:self.input_nodes] == 1)[0]
        child_active_hidden = np.where(child[self.input_nodes:] == 1)[0]
        
        # 부모 모델에서 활성화된 입력과 은닉 노드 확인
        p1_active_inputs = np.where(parent1[:self.input_nodes] == 1)[0]
        p1_active_hidden = np.where(parent1[self.input_nodes:] == 1)[0]
        
        p2_active_inputs = np.where(parent2[:self.input_nodes] == 1)[0]
        p2_active_hidden = np.where(parent2[self.input_nodes:] == 1)[0]
        
        # 가중치 상속을 위한 입력-은닉층 연결 가중치 초기화
        input_hidden_weights = np.random.uniform(-1, 1, (len(child_active_inputs), len(child_active_hidden)))
        hidden_bias = np.random.uniform(-1, 1, len(child_active_hidden))
        
        # 각 연결에 대해 가중치 상속 결정
        for i, input_idx in enumerate(child_active_inputs):
            for j, hidden_idx in enumerate(child_active_hidden):
                # 부모1에서 상속 가능한 연결인지 확인
                p1_input_idx = np.where(p1_active_inputs == input_idx)[0]
                p1_hidden_idx = np.where(p1_active_hidden == hidden_idx)[0]
                
                # 부모2에서 상속 가능한 연결인지 확인
                p2_input_idx = np.where(p2_active_inputs == input_idx)[0]
                p2_hidden_idx = np.where(p2_active_hidden == hidden_idx)[0]
                
                # 상속 로직 구현 (논문에 따른 4가지 경우)
                if len(p1_input_idx) > 0 and len(p1_hidden_idx) > 0 and p1_model is not None:
                    # 부모1에서만 가중치가 있는 경우
                    if len(p2_input_idx) == 0 or len(p2_hidden_idx) == 0 or p2_model is None:
                        p1_weights = p1_model.layers[0].get_weights()[0]
                        input_hidden_weights[i, j] = p1_weights[p1_input_idx[0], p1_hidden_idx[0]]
                    # 부모1과 부모2 모두에서 가중치가 있는 경우
                    elif len(p2_input_idx) > 0 and len(p2_hidden_idx) > 0 and p2_model is not None:
                        p1_weights = p1_model.layers[0].get_weights()[0]
                        p2_weights = p2_model.layers[0].get_weights()[0]
                        # 두 부모의 가중치 평균
                        input_hidden_weights[i, j] = (p1_weights[p1_input_idx[0], p1_hidden_idx[0]] + 
                                                    p2_weights[p2_input_idx[0], p2_hidden_idx[0]]) / 2
                elif len(p2_input_idx) > 0 and len(p2_hidden_idx) > 0 and p2_model is not None:
                    # 부모2에서만 가중치가 있는 경우
                    p2_weights = p2_model.layers[0].get_weights()[0]
                    input_hidden_weights[i, j] = p2_weights[p2_input_idx[0], p2_hidden_idx[0]]
                # 그 외의 경우는 이미 랜덤 초기화된 값 사용
        
        # 은닉층 바이어스 상속
        for j, hidden_idx in enumerate(child_active_hidden):
            p1_hidden_idx = np.where(p1_active_hidden == hidden_idx)[0]
            p2_hidden_idx = np.where(p2_active_hidden == hidden_idx)[0]
            
            if len(p1_hidden_idx) > 0 and p1_model is not None:
                if len(p2_hidden_idx) == 0 or p2_model is None:
                    # 부모1에서만 바이어스가 있는 경우
                    hidden_bias[j] = p1_model.layers[0].get_weights()[1][p1_hidden_idx[0]]
                elif len(p2_hidden_idx) > 0 and p2_model is not None:
                    # 두 부모 모두에서 바이어스가 있는 경우
                    hidden_bias[j] = (p1_model.layers[0].get_weights()[1][p1_hidden_idx[0]] + 
                                     p2_model.layers[0].get_weights()[1][p2_hidden_idx[0]]) / 2
            elif len(p2_hidden_idx) > 0 and p2_model is not None:
                # 부모2에서만 바이어스가 있는 경우
                hidden_bias[j] = p2_model.layers[0].get_weights()[1][p2_hidden_idx[0]]
        
        # 출력층 가중치는 새로 학습하도록 랜덤 초기화
        
        return [input_hidden_weights, hidden_bias]
    
    def evolve(self):
        """
        진화 과정 수행
        
        Returns:
            best_chromosome: 최적의 염색체
            best_fitness: 최적의 적합도
            best_model: 최적의 모델
        """
        # 데이터 분할 (8:1:1 - 훈련:검증:테스트)
        X_train_full, X_test, y_train_full, y_test = train_test_split(self.X, self.y, test_size=0.1, random_state=42)
        X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.111, random_state=42)
        
        # 해집단의 각 염색체 평가
        fitness_values = []
        models = []
        
        for chromosome in self.population:
            fitness, model = self.evaluate_fitness(chromosome, X_train, y_train, X_val, y_val)
            fitness_values.append(fitness)
            models.append(model)
        
        # 최적 염색체 초기화
        best_idx = np.argmax(fitness_values)
        best_chromosome = self.population[best_idx].copy()
        best_fitness = fitness_values[best_idx]
        best_model = models[best_idx]
        
        # 세대 진화
        for generation in range(self.max_generations):
            # 부모 선택
            parent1_idx = self.select_parent(fitness_values)
            parent2_idx = self.select_parent(fitness_values)
            
            # 같은 부모가 선택되지 않도록
            while parent2_idx == parent1_idx:
                parent2_idx = self.select_parent(fitness_values)
            
            parent1 = self.population[parent1_idx].copy()
            parent2 = self.population[parent2_idx].copy()
            
            # 교차 연산
            child = self.crossover(parent1, parent2)
            
            # 돌연변이 연산
            child = self.mutation(child)
            
            # 가중치 상속
            inherited_weights = self.inherit_weights(
                child, parent1, parent2, models[parent1_idx], models[parent2_idx]
            )
            
            # 자식 평가
            child_fitness, child_model = self.evaluate_fitness(child, X_train, y_train, X_val, y_val)
            
            # 대치 (논문에 따라 자식이 부모보다 우수하면 대치, 아니면 가장 나쁜 해와 대치)
            if child_fitness > fitness_values[parent1_idx] or child_fitness > fitness_values[parent2_idx]:
                if fitness_values[parent1_idx] < fitness_values[parent2_idx]:
                    self.population[parent1_idx] = child
                    fitness_values[parent1_idx] = child_fitness
                    models[parent1_idx] = child_model
                else:
                    self.population[parent2_idx] = child
                    fitness_values[parent2_idx] = child_fitness
                    models[parent2_idx] = child_model
            else:
                worst_idx = np.argmin(fitness_values)
                self.population[worst_idx] = child
                fitness_values[worst_idx] = child_fitness
                models[worst_idx] = child_model
            
            # 최적 해 업데이트
            current_best_idx = np.argmax(fitness_values)
            if fitness_values[current_best_idx] > best_fitness:
                best_chromosome = self.population[current_best_idx].copy()
                best_fitness = fitness_values[current_best_idx]
                best_model = models[current_best_idx]
            
            # 세대별 적합도 기록
            self.fitness_history.append(best_fitness)
            
            # 진행 상황 출력 (10세대마다)
            if generation % 10 == 0:
                print(f"세대 {generation}/{self.max_generations}, 최적 적합도: {best_fitness:.4f}")
        
        # 최종 최적 해 평가
        print(f"진화 완료. 최종 최적 적합도: {best_fitness:.4f}")
        
        # 활성화된 노드 수 계산
        active_input_nodes = np.sum(best_chromosome[:self.input_nodes])
        active_hidden_nodes = np.sum(best_chromosome[self.input_nodes:])
        
        print(f"입력층 노드: {active_input_nodes}/{self.input_nodes} ({active_input_nodes/self.input_nodes*100:.1f}%)")
        print(f"은닉층 노드: {active_hidden_nodes}/{self.hidden_nodes} ({active_hidden_nodes/self.hidden_nodes*100:.1f}%)")
        
        return best_chromosome, best_fitness, best_model
    
    def plot_fitness_history(self):
        """
        세대별 적합도 변화 그래프 출력
        """
        plt.figure(figsize=(10, 6))
        plt.plot(range(len(self.fitness_history)), self.fitness_history)
        plt.title('세대별 최적 적합도 변화')
        plt.xlabel('세대')
        plt.ylabel('적합도')
        plt.grid(True)
        plt.show()
    
    def evaluate_model(self, chromosome, model=None):
        """
        최종 모델 평가
        
        Args:
            chromosome: 평가할 염색체
            model: 기존 훈련된 모델 (없으면 새로 훈련)
            
        Returns:
            accuracy: 정확도
        """
        # 데이터 분할
        X_train, X_test, y_train, y_test = train_test_split(self.X, self.y, test_size=0.2, random_state=42)
        
        # 활성화된 입력 노드 확인
        active_input_nodes = np.where(chromosome[:self.input_nodes] == 1)[0]
        active_hidden_nodes = np.sum(chromosome[self.input_nodes:])
        
        if len(active_input_nodes) == 0 or active_hidden_nodes == 0:
            return 0  # 활성화된 노드가 없으면 정확도 0
        
        # 활성화된 입력 특징만 사용
        X_train_pruned = X_train[:, active_input_nodes]
        X_test_pruned = X_test[:, active_input_nodes]
        
        # 모델이 없으면 새로 생성하고 훈련
        if model is None:
            model = Sequential()
            model.add(Dense(active_hidden_nodes, input_dim=len(active_input_nodes), activation='relu'))
            
            if isinstance(y_train, np.ndarray) and len(y_train.shape) > 1 and y_train.shape[1] > 1:
                # 다중 클래스 분류
                model.add(Dense(y_train.shape[1], activation='softmax'))
                loss = 'categorical_crossentropy'
            else:
                # 이진 분류
                model.add(Dense(1, activation='sigmoid'))
                loss = 'binary_crossentropy'
            
            model.compile(loss=loss, optimizer='adam', metrics=['accuracy'])
            
            # 모델 훈련
            model.fit(X_train_pruned, y_train, epochs=100, batch_size=32, verbose=0)
        
        # 테스트 데이터에 대한 정확도 평가
        _, accuracy = model.evaluate(X_test_pruned, y_test, verbose=0)
        
        return accuracy

## 메인 함수 정의 및 실행

In [ ]:
def main():
    """
    메인 함수 - 데이터셋별로 NNPGA 알고리즘 실행
    """
    # 실험에 사용할 데이터셋 목록
    datasets = {
        'balance_scale': (X_balance, y_balance, 4, 12, 3),  # 특징 수, 은닉 노드 수, 클래스 수
        'glass': (X_glass, y_glass, 9, 28, 7),
        'ionosphere': (X_ionosphere, y_ionosphere, 34, 8, 2),
        'iris': (X_iris, y_iris, 4, 12, 3),
        'segmentation': (X_segmentation, y_segmentation, 19, 24, 7),
        'zoo': (X_zoo, y_zoo, 17, 28, 7)
    }
    
    results = {}
    
    # 각 데이터셋에 대해 실험
    for name, (X, y, input_nodes, hidden_nodes, output_nodes) in datasets.items():
        print(f"\n===== {name} 데이터셋 실험 =====")
        
        # 데이터 전처리
        X_scaled, y_encoded = preprocess_data(X, y)
        
        # 노드 가지치기 비율별 실험
        pruning_rates = [0.04, 0.08, 0.14, 0.20, 0.25, 0.30]
        pruning_results = []
        
        for rate in pruning_rates:
            print(f"\n## 가지치기 비율: {rate*100}% ##")
            
            # NNPGA 알고리즘 실행
            nnpga = NNPGA(X_scaled, y_encoded, input_nodes, hidden_nodes, output_nodes, 
                          population_size=20, max_generations=50)  # 논문에서는 150세대지만 시간 절약을 위해 50세대로 감소
            
            best_chromosome, best_fitness, best_model = nnpga.evolve()
            
            # 최종 모델 정확도 평가
            accuracy = nnpga.evaluate_model(best_chromosome, best_model)
            
            pruning_results.append({
                'pruning_rate': rate,
                'fitness': best_fitness,
                'accuracy': accuracy,
                'active_input': np.sum(best_chromosome[:input_nodes]),
                'active_hidden': np.sum(best_chromosome[input_nodes:])
            })
            
            print(f"가지치기 비율 {rate*100}%에서의 정확도: {accuracy:.4f}")
            
            # 적합도 변화 그래프
            nnpga.plot_fitness_history()
        
        results[name] = pruning_results
    
    # 결과 요약
    print("\n===== 실험 결과 요약 =====")
    for name, pruning_results in results.items():
        print(f"\n{name} 데이터셋:")
        print("가지치기 비율 | 적합도 | 정확도 | 활성 입력 노드 | 활성 은닉 노드")
        print("-" * 70)
        
        for result in pruning_results:
            print(f"{result['pruning_rate']*100:>13.1f}% | {result['fitness']:.4f} | {result['accuracy']:.4f} | "
                  f"{result['active_input']:>14} | {result['active_hidden']:>14}")
    
    return results

if __name__ == "__main__":
    main()